# Cut a county inventory to a city for R2D

This notebook takes a synthetic building inventory of a county, keeps the buildings of one city, and writes one possible world of them with integer ids, the asset type `building` and the class `RES3` for multifamily housing, the form that SimCenter's R2D tool takes as input. It can also write all the possible worlds to one file. The example uses the county file of Alameda County, California, and the city of Berkeley.

A county file holds ten possible worlds: ten versions of the inventory that differ only in the values BRAILS++ drew at random, such as the year a building was built. The notebook needs no Census API key. It downloads one file, the Census Bureau's boundaries of the places in the city's state.

## Name the county file, the city and the world

The county files are shared as zip archives. Download one, unzip it, and put the GeoJSON file in the folder of this notebook, or give its full path below. The name of a county file holds the county's FIPS code, its name and the date of its release, as in `06001_Alameda_261120.geojson`. The date changes with every release, so write the name of the file you downloaded. On Windows, write a full path with forward slashes, as in `'D:/data/06001_Alameda_261120.geojson'`, since Python reads a backslash followed by some letters as a special character. Any inventory that BRAILS++ wrote with `write_to_geojson`, or with `write` of a compilation, can take the county file's place. Give the city with its state, as in `'Berkeley, CA'`, and the world to write, an index from 0 to 9, or `None` for all of them.

In [ ]:
# To cut another city from its county file, change these values.
COUNTY_FILE = '06001_Alameda_261120.geojson'
CITY = 'Berkeley, CA'
WORLD = 3

## Keep the buildings of the city

`filter_to_region` looks up the boundary of the city in the Census Bureau's TIGER/Line file of places and keeps each building whose footprint has its center inside that boundary. The boundary keeps its holes, so a city with an enclave, such as Oakland with Piedmont inside it, keeps none of the enclave's buildings. The function reads the county file one building at a time, so the memory it needs grows with the buildings it keeps, not with the size of the file.

In [ ]:
from brails.utils.region_filter import filter_to_region

city_inventory = filter_to_region(COUNTY_FILE, CITY)

## See how the possible worlds are stored

A property holds one value where all ten possible worlds agree, and a list of ten values, for worlds 0 to 9 in that order, where they differ. A `None` in a list means that the building has no value in that world. A building lacks a property that does not apply to it, such as `HeightClass` for a wood building (W1, W2), and a table of many buildings shows such a property as `NaN`. The properties of three buildings:

In [ ]:
three = city_inventory.get_asset_ids()[:3]
for asset_id in three:
    print(asset_id)
    for name, value in city_inventory.inventory[asset_id].features.items():
        print('   ', name, value)

## Take one possible world

`get_world_realization` returns an inventory of one possible world, in which each property holds the value of that world. The same three buildings in world 3:

In [ ]:
world_3 = city_inventory.get_world_realization(3)
features, _, _ = world_3.get_dataframe()
features.loc[three].T

## Give the RES3 subtypes the class RES3

R2D's Hazus earthquake damage and loss models do not distinguish the subtypes `RES3A` to `RES3F` of multifamily housing, and some R2D Python environments refuse them and leave those buildings out of the results. The cell below therefore gives the class `RES3` to the `RES3A` to `RES3F` buildings of the inventory that the next cell writes, in every world that inventory holds. Their number of units stays in `NumberOfUnits`. The inventory is world `WORLD`, taken afresh so that the cell can be run twice, or, when `WORLD` is `None`, `city_inventory` itself, which the cell changes: a second run then finds no subtype left and prints 0. The county file keeps the subtypes, and so does `city_inventory` when the file holds one world.

In [ ]:
r2d_inventory = (
    city_inventory if WORLD is None else city_inventory.get_world_realization(WORLD)
)
RES3_SUBTYPES = {'RES3A', 'RES3B', 'RES3C', 'RES3D', 'RES3E', 'RES3F'}
replaced = 0
for asset in r2d_inventory.inventory.values():
    value = asset.features.get('OccupancyClass')
    if isinstance(value, list):
        classes = ['RES3' if v in RES3_SUBTYPES else v for v in value]
    else:
        classes = 'RES3' if value in RES3_SUBTYPES else value
    if classes != value:
        asset.features['OccupancyClass'] = classes
        replaced += 1
print(replaced, 'buildings take the class RES3 in place of a subtype')

## Write the file for R2D

The cell below writes world `WORLD` to a file in the folder of this notebook, named after the city and the world. `integer_ids=True` numbers the buildings from 1 and keeps the id of each building in the property `unique_id`, and `asset_type='building'` gives each building the property `type`, by which R2D groups the assets. This example prepares a deterministic inventory that runs in R2D as one world per file: to run another world, set `WORLD` at the top to its index and run the notebook again. With `WORLD = None`, the file holds all the possible worlds.

On Windows, close the file in other programs, such as QGIS or R2D, before you run the cell again: a file open in another program cannot be replaced.

In [ ]:
world_label = 'all_worlds' if WORLD is None else f'world_{WORLD}'
r2d = r2d_inventory.write_to_geojson(
    CITY.split(',')[0].replace(' ', '_') + f'_{world_label}_R2D.geojson',
    integer_ids=True,
    asset_type='building',
)
first = r2d['features'][0]
print(
    'The first building is numbered',
    first['id'],
    'and keeps its id',
    first['properties']['unique_id'],
)

## A region in two county files

For a region that spans two county files, give both files in a list. `filter_to_region` keeps the buildings inside the region from each file and joins them into one inventory. The region can be a polygon in longitude and latitude, such as this rectangle across the line between Alameda and Contra Costa counties. The cell needs the Contra Costa file too, so it is not run here:

```python
import shapely

area = shapely.box(-122.31, 37.88, -122.27, 37.91)
both = filter_to_region(
    ['06001_Alameda_261120.geojson', '06013_Contra_Costa_261120.geojson'], area
)
```